In [1]:
import os

DELTA_PATH = "../data/raw/delta"

print("FILES INSIDE data/raw/delta:")
print("="*60)
ext_map = {}
for root, dirs, files in os.walk(DELTA_PATH):
    for fname in files:
        full_path = os.path.join(root, fname)
        ext = os.path.splitext(fname)[1].lower()
        ext_map.setdefault(ext, []).append(full_path)
        print(full_path)

print("\n" + "="*60)
print("SUMMARY BY EXTENSION:")
for ext, paths in sorted(ext_map.items(), key=lambda x: -len(x[1])):
    total_mb = sum(os.path.getsize(p) for p in paths) / 1e6
    print(f"  {ext:15s} : {len(paths):4d} files, {total_mb:.1f} MB")

FILES INSIDE data/raw/delta:
../data/raw/delta\.git-blame-ignore-revs
../data/raw/delta\.gitattributes
../data/raw/delta\.gitignore
../data/raw/delta\.gitlab-ci.yml
../data/raw/delta\.pre-commit-config.yaml
../data/raw/delta\.readthedocs.yaml
../data/raw/delta\.typos.toml
../data/raw/delta\CHANGELOG.md
../data/raw/delta\CODE_OF_CONDUCT.md
../data/raw/delta\CONTRIBUTING.md
../data/raw/delta\GOVERNANCE.md
../data/raw/delta\LICENSE
../data/raw/delta\pixi.lock
../data/raw/delta\pyproject.toml
../data/raw/delta\README.md
../data/raw/delta\RELEASE.md
../data/raw/delta\.git\config
../data/raw/delta\.git\description
../data/raw/delta\.git\HEAD
../data/raw/delta\.git\index
../data/raw/delta\.git\packed-refs
../data/raw/delta\.git\hooks\applypatch-msg.sample
../data/raw/delta\.git\hooks\commit-msg.sample
../data/raw/delta\.git\hooks\fsmonitor-watchman.sample
../data/raw/delta\.git\hooks\post-update.sample
../data/raw/delta\.git\hooks\pre-applypatch.sample
../data/raw/delta\.git\hooks\pre-commit.

In [2]:
import tifffile
import numpy as np

tif_files = ext_map.get('.tif', [])
print(f"Total .tif files found: {len(tif_files)}")
print("\nFirst 5 TIF files and their shapes:")
for path in tif_files[:5]:
    img = tifffile.imread(path)
    print(f"  {os.path.basename(path)}")
    print(f"    shape: {img.shape}, dtype: {img.dtype}")
    print(f"    min: {img.min()}, max: {img.max()}")

Total .tif files found: 79

First 5 TIF files and their shapes:
  angle_+0.53.tif
    shape: (520, 696), dtype: uint16
    min: 1387, max: 11402
  angle_-0.32.tif
    shape: (520, 696), dtype: uint16
    min: 0, max: 65535
  img_frame1.tif
    shape: (520, 696), dtype: uint16
    min: 1297, max: 16383
  img_frame2.tif
    shape: (520, 696), dtype: uint16
    min: 1325, max: 16383
  reference.tif
    shape: (520, 696), dtype: uint16
    min: 1089, max: 16383


In [3]:
import scipy.io

mat_files = ext_map.get('.mat', [])
print(f"Total .mat files found: {len(mat_files)}")
print("\nInspecting all MAT files:")
for path in mat_files:
    print(f"\n{'='*50}")
    print(f"File: {path}")
    try:
        mat = scipy.io.loadmat(path)
        keys = [k for k in mat.keys() if not k.startswith('_')]
        print(f"  Keys: {keys}")
        for k in keys:
            v = mat[k]
            print(f"  '{k}' → type: {type(v).__name__}, shape: {getattr(v,'shape','N/A')}")
            if hasattr(v, 'shape') and v.ndim <= 2 and v.size < 50:
                print(f"    values: {v}")
    except Exception as e:
        print(f"  Could not read with scipy.io: {e}")
        print(f"  → Run: pip install mat73, then retry with mat73.loadmat(path)")

Total .mat files found: 4

Inspecting all MAT files:

File: ../data/raw/delta\tests\data\movie_2D_nd2\test_expected_results\Position000000.mat
  Keys: ['moviedimensions', 'tifffile', 'proc', 'res']
  'moviedimensions' → type: ndarray, shape: (1, 4)
    values: [[520 696   1  10]]
  'tifffile' → type: ndarray, shape: (1,)
    values: ['/home/virgile/src/DeLTA/tests/data/movie_nd2/test.nd2']
  'proc' → type: ndarray, shape: (1, 1)
    values: [[(array([[0]]), array([], shape=(2, 0), dtype=float64), array([[  0.,   0., 696., 520.]]))]]
  'res' → type: ndarray, shape: (1, 1)
    values: [[array([[(array([[[0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0],
                  ...,
                  [0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0]],

                 [[0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0

In [5]:
import nd2

nd2_files = ext_map.get('.nd2', [])
print(f"Total .nd2 files found: {len(nd2_files)}")

for path in nd2_files:
    print(f"\nFile: {path}")
    with nd2.ND2File(path) as f:
        print(f"  Shape : {f.shape}")
        print(f"  Sizes : {f.sizes}")          # axis name → size mapping
        print(f"  Dtype : {f.dtype}")
        print(f"  Size  : {os.path.getsize(path)/1e6:.1f} MB")

Total .nd2 files found: 1

File: ../data/raw/delta\tests\data\movie_2D_nd2\test.nd2
  Shape : (10, 520, 696)
  Sizes : {'T': 10, 'Y': 520, 'X': 696}
  Dtype : uint16
  Size  : 7.6 MB


In [6]:
csv_files = ext_map.get('.csv', [])
json_files = ext_map.get('.json', [])
txt_files = ext_map.get('.txt', [])

print(f"CSV files in delta: {len(csv_files)}")
for p in csv_files: print(f"  {p}")

print(f"\nJSON files in delta: {len(json_files)}")
for p in json_files: print(f"  {p}")

print(f"\nTXT files in delta: {len(txt_files)}")
for p in txt_files: print(f"  {p}")

CSV files in delta: 0

JSON files in delta: 0

TXT files in delta: 0


In [7]:
import scipy.io

mat_files = ext_map.get('.mat', [])
print(f"Total .mat files found: {len(mat_files)}")
print("\nInspecting first 3 MAT files:")

for path in mat_files[:3]:
    print(f"\n{'='*50}")
    print(f"File: {path}")
    try:
        mat = scipy.io.loadmat(path)
        keys = [k for k in mat.keys() if not k.startswith('_')]
        print(f"  Keys: {keys}")
        for k in keys:
            v = mat[k]
            print(f"  '{k}' → type: {type(v).__name__}, shape: {getattr(v,'shape','N/A')}")
            if hasattr(v, 'shape') and v.ndim <= 2 and v.size < 50:
                print(f"    values: {v}")
    except Exception as e:
        print(f"  Could not read with scipy.io: {e}")
        print(f"  Try: import mat73; mat73.loadmat(path)")

Total .mat files found: 4

Inspecting first 3 MAT files:

File: ../data/raw/delta\tests\data\movie_2D_nd2\test_expected_results\Position000000.mat
  Keys: ['moviedimensions', 'tifffile', 'proc', 'res']
  'moviedimensions' → type: ndarray, shape: (1, 4)
    values: [[520 696   1  10]]
  'tifffile' → type: ndarray, shape: (1,)
    values: ['/home/virgile/src/DeLTA/tests/data/movie_nd2/test.nd2']
  'proc' → type: ndarray, shape: (1, 1)
    values: [[(array([[0]]), array([], shape=(2, 0), dtype=float64), array([[  0.,   0., 696., 520.]]))]]
  'res' → type: ndarray, shape: (1, 1)
    values: [[array([[(array([[[0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0],
                  ...,
                  [0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0]],

                 [[0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 

In [13]:
import os, tifffile, numpy as np

SPECIES = "bsubtilis"  # then change to "bsubtilis", then "saureus"
SPECIES_PATH = f"../data/raw/deepbacs/{SPECIES}"

print(f"DEEPBACS — {SPECIES.upper()}")
print("="*60)

ext_map = {}
for root, dirs, files in os.walk(SPECIES_PATH):
    for fname in files:
        full_path = os.path.join(root, fname)
        ext = os.path.splitext(fname)[1].lower()
        ext_map.setdefault(ext, []).append(full_path)

# File counts and sizes
for ext, paths in sorted(ext_map.items(), key=lambda x: -len(x[1])):
    total_mb = sum(os.path.getsize(p) for p in paths) / 1e6
    print(f"  {ext:15s} : {len(paths):4d} files, {total_mb:.1f} MB")

# Look for any label/annotation files
print("\nLABEL/ANNOTATION FILES:")
for ext in ['.csv', '.json', '.txt', '.xml']:
    for p in ext_map.get(ext, []):
        print(f"  {p}")
        if ext == '.csv':
            import pandas as pd
            try:
                df = pd.read_csv(p, nrows=5)
                print(f"  Columns: {list(df.columns)}")
                print(df.head(3).to_string())
            except:
                print("  Could not parse as CSV")

# Sample image shapes
print("\nSAMPLE IMAGE SHAPES:")
for ext in ['.tif', '.tiff', '.png']:
    for p in ext_map.get(ext, [])[:3]:
        img = tifffile.imread(p)
        print(f"  {os.path.basename(p)} → shape: {img.shape}, dtype: {img.dtype}")
        

DEEPBACS — BSUBTILIS
  .tif            :  284 files, 247.7 MB
  .png            :  180 files, 24.3 MB
  .txt            :    3 files, 0.0 MB

LABEL/ANNOTATION FILES:
  ../data/raw/deepbacs/bsubtilis\CARE_U-Net_dataset\Notes_CARE_U-Net.txt
  ../data/raw/deepbacs/bsubtilis\pix2pix_dataset\Notes_pix2pix.txt
  ../data/raw/deepbacs/bsubtilis\SplineDist_dataset\Notes.txt

SAMPLE IMAGE SHAPES:
  test_1.tif → shape: (1024, 1024), dtype: uint8
  test_10.tif → shape: (1024, 1024), dtype: uint8
  test_2.tif → shape: (1024, 1024), dtype: uint8


TiffFileError: not a TIFF file: header=b'\x89PNG'

In [9]:
with open("../data/raw/deepbacs/ecoli/live-cell_data/Notes.txt") as f:
    print(f.read())

For the segmentation task, the time series were preprocessed by scaling 2x without interpolation.

This was performed to be able to generate better segmentation masks and increase object size, as small object sizes can affect network performance.
Both unscaled and scaled time series are provided here.


In [10]:
for root, dirs, files in os.walk("../data/raw/deepbacs/ecoli"):
    level = root.replace("../data/raw/deepbacs/ecoli", "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in files[:3]:
        print("  " * (level+1) + f)

ecoli/
  live-cell_data/
    Notes.txt
    time_series_raw/
      series_0.tif
      series_1.tif
      series_2.tif
    time_series_scaled/
      series_0.tif
      series_1.tif
      series_2.tif
  test/
    brightfield/
      pos2_fr_1.tif
      pos2_fr_20.tif
      pos2_fr_40.tif
    masks_binary/
      pos2_fr_1.tif
      pos2_fr_20.tif
      pos2_fr_40.tif
    masks_RoiMap/
      pos2_fr_1.tif
      pos2_fr_20.tif
      pos2_fr_40.tif
  train/
    brightfield/
      pos0_frame_26.tif
      pos0_frame_56.tif
      pos0_frame_6.tif
    masks_binary/
      pos0_frame_26.tif
      pos0_frame_56.tif
      pos0_frame_6.tif
    masks_RoiMap/
      pos0_frame_26.tif
      pos0_frame_56.tif
      pos0_frame_6.tif


In [11]:
import tifffile
import numpy as np

mask = tifffile.imread("../data/raw/deepbacs/ecoli/train/masks_RoiMap/pos0_frame_26.tif")
print(f"Shape: {mask.shape}, dtype: {mask.dtype}")
print(f"Unique values (cell IDs): {np.unique(mask)}")
print(f"Number of cells in this frame: {len(np.unique(mask)) - 1}")  # subtract background (0)

Shape: (1024, 1024), dtype: uint8
Unique values (cell IDs): [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68]
Number of cells in this frame: 68


In [14]:
import cv2

print("\nSAMPLE IMAGE SHAPES:")

# TIF files
for p in ext_map.get('.tif', [])[:3]:
    try:
        img = tifffile.imread(p)
        print(f"  {os.path.basename(p)} → shape: {img.shape}, dtype: {img.dtype}")
    except Exception as e:
        print(f"  {os.path.basename(p)} → ERROR: {e}")

# PNG files
for p in ext_map.get('.png', [])[:3]:
    img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
    print(f"  {os.path.basename(p)} → shape: {img.shape}, dtype: {img.dtype}")


SAMPLE IMAGE SHAPES:
  test_1.tif → shape: (1024, 1024), dtype: uint8
  test_10.tif → shape: (1024, 1024), dtype: uint8
  test_2.tif → shape: (1024, 1024), dtype: uint8
  test_1.png → shape: (1024, 1024, 3), dtype: uint8
  test_10.png → shape: (1024, 1024, 3), dtype: uint8
  test_2.png → shape: (1024, 1024, 3), dtype: uint8


In [15]:
for ext in ['.txt']:
    for p in ext_map.get(ext, []):
        print(f"\n{'='*40}")
        print(f"FILE: {p}")
        with open(p) as f:
            print(f.read())


FILE: ../data/raw/deepbacs/bsubtilis\CARE_U-Net_dataset\Notes_CARE_U-Net.txt
CARE and U-Net do not require images in which instances are represented by unique gray values.
Thus, we provide binary masks for training of these networks.

Note that the output of CARE and U-Net is not a binary image, but has to be converted into one via thresholding.

FILE: ../data/raw/deepbacs/bsubtilis\pix2pix_dataset\Notes_pix2pix.txt
pix2pix requires PNG images, so this dataset contains the PNG versions of train and test images used for StarDist.

Hence, the output is also a PNG image which has to be binarized in order to obtain the semantic segmentation.

FILE: ../data/raw/deepbacs/bsubtilis\SplineDist_dataset\Notes.txt
SplineDist is computationally more expensive than StarDist.
We thus provide a reduced dataset with smaller images patches (32 image pairs @ 512 x 512 px). 
This reduces the memory required during the training procedure.

This does not affect prediction the same amount, hence test image

In [16]:
for root, dirs, files in os.walk("../data/raw/deepbacs/bsubtilis"):
    level = root.replace("../data/raw/deepbacs/bsubtilis", "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in files[:3]:
        print("  " * (level+1) + f)

bsubtilis/
  CARE_U-Net_dataset/
    Notes_CARE_U-Net.txt
    test/
      fluorescence/
        test_1.tif
        test_10.tif
        test_2.tif
      masks/
        0.tif
        118.tif
        158.tif
    train/
      fluorescence/
        train_1.tif
        train_10.tif
        train_11.tif
      masks/
        200104_1mW-TIRF-30_CDM_1_3_MMStack_Pos0.ome-0001.tif
        200104_1mW-TIRF-30_CDM_1_3_MMStack_Pos0.ome-0020.tif
        200104_1mW-TIRF-30_CDM_1_3_MMStack_Pos0.ome-0040.tif
  pix2pix_dataset/
    Notes_pix2pix.txt
    test/
      fluorescence/
        test_1.png
        test_10.png
        test_2.png
      masks/
        test_1.png
        test_10.png
        test_2.png
    train/
      fluorescence/
        train_1.png
        train_10.png
        train_11.png
      masks/
        train_1.png
        train_10.png
        train_11.png
  SplineDist_dataset/
    Notes.txt
    test/
      fluorescence/
        test_1.tif
        test_10.tif
        test_2.tif
      masks/
 

In [17]:
mask = tifffile.imread("../data/raw/deepbacs/bsubtilis/StarDist_dataset/test/masks/test_1.tif")
print(f"Shape: {mask.shape}, dtype: {mask.dtype}")
print(f"Unique values: {np.unique(mask)}")
print(f"Number of cells: {len(np.unique(mask)) - 1}")

Shape: (1024, 1024), dtype: uint8
Unique values: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27]
Number of cells: 27


In [18]:
import os
import tifffile
import numpy as np
import cv2

SPECIES = "staph"
SPECIES_PATH = f"../data/raw/deepbacs/{SPECIES}"

print(f"DEEPBACS — {SPECIES.upper()}")
print("="*60)

# Build extension map
ext_map = {}
for root, dirs, files in os.walk(SPECIES_PATH):
    for fname in files:
        full_path = os.path.join(root, fname)
        ext = os.path.splitext(fname)[1].lower()
        ext_map.setdefault(ext, []).append(full_path)

# File counts and sizes
for ext, paths in sorted(ext_map.items(), key=lambda x: -len(x[1])):
    total_mb = sum(os.path.getsize(p) for p in paths) / 1e6
    print(f"  {ext:15s} : {len(paths):4d} files, {total_mb:.1f} MB")

# Label/annotation files
print("\nLABEL/ANNOTATION FILES:")
for ext in ['.csv', '.json', '.txt', '.xml']:
    for p in ext_map.get(ext, []):
        print(f"  {p}")
        if ext == '.csv':
            import pandas as pd
            try:
                df = pd.read_csv(p, nrows=5)
                print(f"  Columns: {list(df.columns)}")
                print(df.head(3).to_string())
            except:
                print("  Could not parse as CSV")

# Read all text files
print("\nTEXT FILE CONTENTS:")
for p in ext_map.get('.txt', []):
    print(f"\n{'='*40}")
    print(f"FILE: {p}")
    with open(p) as f:
        print(f.read())

# Sample image shapes
print("\nSAMPLE IMAGE SHAPES:")
for p in ext_map.get('.tif', [])[:3]:
    try:
        img = tifffile.imread(p)
        print(f"  {os.path.basename(p)} → shape: {img.shape}, dtype: {img.dtype}")
    except Exception as e:
        print(f"  {os.path.basename(p)} → ERROR: {e}")

for p in ext_map.get('.png', [])[:3]:
    img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
    print(f"  {os.path.basename(p)} → shape: {img.shape}, dtype: {img.dtype}")

# Folder structure
print("\nFOLDER STRUCTURE:")
for root, dirs, files in os.walk(SPECIES_PATH):
    level = root.replace(SPECIES_PATH, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in files[:3]:
        print("  " * (level+1) + f)

# Check one mask for instance vs binary
print("\nMASK INSPECTION:")
all_tifs = ext_map.get('.tif', [])
mask_files = [p for p in all_tifs if 'mask' in p.lower()]
if mask_files:
    mask = tifffile.imread(mask_files[0])
    print(f"  File: {mask_files[0]}")
    print(f"  Shape: {mask.shape}, dtype: {mask.dtype}")
    print(f"  Unique values: {np.unique(mask)}")
    print(f"  Number of cells: {len(np.unique(mask)) - 1}")
else:
    print("  No mask files found with 'mask' in filename")

DEEPBACS — STAPH
  .tif            :  160 files, 33.5 MB
  .zip            :   12 files, 1.3 MB
  .txt            :    1 files, 0.0 MB

LABEL/ANNOTATION FILES:
  ../data/raw/deepbacs/staph\Notes.txt

TEXT FILE CONTENTS:

FILE: ../data/raw/deepbacs/staph\Notes.txt
For the S. aureus dataset, we provide the full images (512 x 512 pixels) as well as quartered images (256 x 256 px).
Brightfield and fluorescence images were take from the same regions of interest.
Hence, mask images are similar and are just added to each folder to prevent any confusion.

SAMPLE IMAGE SHAPES:
  JE2NileRed_oilp22_PMP_101220_007.tif → shape: (512, 512), dtype: uint16
  JE2NileRed_oilp22_PMP_101220_008.tif → shape: (512, 512), dtype: uint16
  JE2NileRed_oilp22_PMP_101220_009.tif → shape: (512, 512), dtype: uint16

FOLDER STRUCTURE:
staph/
  Notes.txt
  brightfield_dataset/
    test/
      brightfield/
        JE2NileRed_oilp22_PMP_101220_007.tif
        JE2NileRed_oilp22_PMP_101220_008.tif
        JE2NileRed_oilp

In [1]:
# Cell A — MAT file inspection
import scipy.io, os

DELTA_PATH = "../data/raw/delta"
mat_files = []
for root, dirs, files in os.walk(DELTA_PATH):
    for f in files:
        if f.endswith('.mat'):
            mat_files.append(os.path.join(root, f))

for path in mat_files:
    print(f"\n{'='*55}")
    print(f"FILE: {path}")
    try:
        mat = scipy.io.loadmat(path)
        keys = [k for k in mat.keys() if not k.startswith('_')]
        print(f"Keys: {keys}")
        for k in keys:
            v = mat[k]
            shape = getattr(v, 'shape', 'N/A')
            dtype = getattr(v, 'dtype', 'N/A')
            print(f"  '{k}' → shape: {shape}, dtype: {dtype}")
            if hasattr(v, 'shape') and v.ndim <= 2 and v.size < 30:
                print(f"    sample values: {v}")
    except NotImplementedError:
        # MAT v7.3 — need mat73
        print("  scipy failed (v7.3 format). Installing mat73...")
        # run: pip install mat73
        import mat73
        mat = mat73.loadmat(path)
        print(f"  Keys (mat73): {list(mat.keys())}")


FILE: ../data/raw/delta\tests\data\movie_2D_nd2\test_expected_results\Position000000.mat
Keys: ['moviedimensions', 'tifffile', 'proc', 'res']
  'moviedimensions' → shape: (1, 4), dtype: int64
    sample values: [[520 696   1  10]]
  'tifffile' → shape: (1,), dtype: <U53
    sample values: ['/home/virgile/src/DeLTA/tests/data/movie_nd2/test.nd2']
  'proc' → shape: (1, 1), dtype: [('rotation', 'O'), ('XYdrift', 'O'), ('chambers', 'O')]
    sample values: [[(array([[0]]), array([], shape=(2, 0), dtype=float64), array([[  0.,   0., 696., 520.]]))]]
  'res' → shape: (1, 1), dtype: object
    sample values: [[array([[(array([[[0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0],
                  ...,
                  [0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0]],

                 [[0, 0, 0, ..., 0, 0, 0],
                  [0, 0, 0, ..., 0, 0, 0],
                  [0

In [2]:
# Cell B — NetCDF inspection
import netCDF4 as nc   # pip install netCDF4 if missing
import os

DELTA_PATH = "../data/raw/delta"
nc_files = []
for root, dirs, files in os.walk(DELTA_PATH):
    for f in files:
        if f.endswith('.nc'):
            nc_files.append(os.path.join(root, f))

for path in nc_files:
    print(f"\n{'='*55}")
    print(f"FILE: {os.path.basename(path)}")
    ds = nc.Dataset(path)
    print(f"  Dimensions: {dict(ds.dimensions)}")
    print(f"  Variables:")
    for vname, var in ds.variables.items():
        print(f"    '{vname}' → shape: {var.shape}, dtype: {var.dtype}")
        if var.size < 20:
            print(f"      values: {var[:]}")
    ds.close()

ModuleNotFoundError: No module named 'netCDF4'